In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

print("正在載入 Qwen2.5-Coder 本地模型...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto" # 自動使用 GPU 或 CPU
)

prompt = """請將輸入的文本按照順序切分成 JSON 陣列，類型包含："code"、"text"、"question"。
請直接輸出標準 JSON，格式如下：
{
  "segments": [
    {"segment_type": "text", "content": "..."},
    {"segment_type": "code", "content": "..."},
    {"segment_type": "question", "content": "..."}
  ]
}

待處理文字：
"""

mixed_input = """
#include "BattleSystem.h"
#include <iostream>
#include <map>
#include "skill.h"
#include "item.h"
using namespace std;

// 定义静态敌人数据库
static map<string, EnemyData> enemyDatabase;

void initializeEnemyDatabase() {
    enemyDatabase["Goblin"] = {"Goblin", 50, 10};
    enemyDatabase["Orc"] = {"Orc", 80, 15};
    enemyDatabase["Dragon"] = {"Dragon", 200, 30};
    // 添加更多敌人数据...
}

EnemyData getEnemyData(const string& enemyName) {
    if (enemyDatabase.find(enemyName) != enemyDatabase.end()) {
        return enemyDatabase[enemyName];
    } else {
        cerr << "Error: Enemy " << enemyName << " not found in database!" << endl;
        return {"Unknown", 0, 0};
    }
}
    // 检查战斗结果
    if (player.hp <= 0) {
        cout << "You have been defeated by the " << enemy.name << "!" << endl;
    }
}
"""

messages = [
    {"role": "system", "content": "你是一個精準的文本與程式碼結構化解析助手。請僅輸出 JSON 格式。"},
    {"role": "user", "content": prompt + mixed_input}
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# 推理生成
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512,
    temperature=0.1 # 降低隨機性
)

response = tokenizer.batch_decode(
    [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)],
    skip_special_tokens=True
)[0]

print("\n【高準確率切分結果】:")
print(response)